> **기록 표시 (2026-11-05):** 이 노트북은 항목 4 추출 당시의 검수 대기 기록이다. 이후 검수 승인 → 승인 파일 등록 → 노트북 23에서 행동 추가 적재 순서로 진행되었다. 현재 상태는 `일정관리_prevention_legal.md`를 본다.


# 22. 93쪽 항목 4 추출 (검수 대기)

- 목적: 항목 4(MMR 2차 미완료자 안내, 블록 `kdca_measles_2026:p0093:b04`)를 v2 추출 프롬프트로 한 번 추출한다.
- 방법: 04 v2 노트북의 스키마(셀 4)와 프롬프트(셀 6)를 그대로 가져온다. 프롬프트를 다시 쓰지 않는다.
- 저장: `steps_pending_b04.jsonl`에 `review_status = unreviewed`로만 쓴다. 승인 파일(`steps_raw_v2.jsonl`)은 건드리지 않는다.
- 승인은 사용자가 한다. 승인 전에는 적재하지 않는다.

In [1]:
import json, pathlib
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# 04 v2 노트북의 스키마(셀 4)와 프롬프트(셀 6)를 그대로 실행해 가져온다
src = json.load(open("04_추출프롬프트_v2.ipynb", encoding="utf-8"))
ns = {}
for idx in (4, 6):
    exec("".join(src["cells"][idx]["source"]), ns)
print("프롬프트 길이:", len(ns["PROMPT_V2"]))

# 대상 블록 하나 (항목 4)
BLOCK_ID = "kdca_measles_2026:p0093:b04"
blocks = [json.loads(l) for l in open("../../data/interim/prevention_legal/blocks.jsonl", encoding="utf-8") if l.strip()]
block = next(b for b in blocks if b["chunk_id"] == BLOCK_ID)
print(block["text"])


스키마 준비 완료
648 자 프롬프트 v2 준비
프롬프트 길이: 648
4. 교내 **MMR(홍역** **유행성이하선염** **풍진) 2차 접종 미완료자** 가 있는 경우 감염 예방을 위해 **예방접종을 완료할 필요가 있음을 안내**


In [2]:
load_dotenv(find_dotenv(usecwd=True), override=False)   # API 키는 .env 에서 읽는다 (값은 출력하지 않음)
model = init_chat_model("openai:gpt-6-luna", reasoning_effort="none")
extractor = model.with_structured_output(ns["ExtractionResult"])

section = block.get("section") or "(소제목 없음)"
human = ("문서: " + block["document_id"] + " / " + str(block["pdf_page"]) + "쪽 / 블록 " + block["chunk_id"]
         + "\n소제목(문맥): " + section + "\n\n원문:\n" + block["text"])
out = extractor.invoke([("system", ns["PROMPT_V2"]), ("human", human)])
for a in out.actions:
    print(a.model_dump())


{'action_text': 'MMR 2차 접종 미완료자에게 예방접종을 완료할 필요가 있음을 안내한다.', 'stage': 'UNSPECIFIED', 'scope': 'NAMED_DISEASES', 'when_text': '교내 MMR(홍역 유행성이하선염 풍진) 2차 접종 미완료자가 있는 경우', 'timing_text': None, 'modality': 'UNSPECIFIED', 'role_name': '학교[담당교사]', 'basis': 'EXPLICIT', 'evidence_quote': '4. 교내 **MMR(홍역** **유행성이하선염** **풍진) 2차 접종 미완료자** 가 있는 경우 감염 예방을 위해 **예방접종을 완료할 필요가 있음을 안내**'}


In [3]:
import re
import pathlib

def compact(s):
    # 공백, 강조 표시(*), 대괄호를 뺀 비교용 글
    return re.sub(r"[\s*\[\]]", "", s)

rows = []
for n, a in enumerate(out.actions, start=1):
    d = a.model_dump()
    quote_ok = compact(d["evidence_quote"]) in compact(block["text"])
    rows.append({
        "run_id": "pending_b04", "chunk_id": BLOCK_ID, "action_no": str(n), "status": "ok",
        "action_text": d["action_text"], "stage": d["stage"], "scope": d["scope"], "modality": d["modality"],
        "when_text": d["when_text"], "timing_text": d["timing_text"], "role_name": d["role_name"],
        "basis": d["basis"], "evidence_quote": d["evidence_quote"],
        "quote_check": "블록 일치" if quote_ok else "블록 불일치",
        "context_section": section, "review_status": "unreviewed",
        "model": "gpt-6-luna", "prompt_version": "v2", "review_note": "검수 대기 (사용자 승인 전)",
    })
    print(n, "근거 확인:", rows[-1]["quote_check"])

out_path = pathlib.Path("../../data/interim/prevention_legal/steps_pending_b04.jsonl")
out_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
print("저장:", out_path.name, len(rows), "행 (검수 대기)")


1 근거 확인: 블록 일치
저장: steps_pending_b04.jsonl 1 행 (검수 대기)
